[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://github.com/Jheff-AIML/data-and-infrastructure-2026_project_m1/blob/notebooks/healthcare_fraud_milestone_1.ipynb)

In [ ]:
# Basic imports
import os
import requests

In [ ]:
## 1. Clone the repository
REPO_URL = "https://github.com/Jheff-AIML/data-and-infrastructure-2026_project_m1.git"
REPO_NAME = "data-and-infrastructure-2026_project_m1"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

## 2. Change the working directory to the cloned repository
%cd {REPO_NAME}

## 3. Install the requirements
if os.path.exists("requirements.txt"):
    !pip install -r requirements.txt
else:
    print("requirements.txt not found!")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import klib
from google.cloud import storage
from google.cloud import bigquery
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.metrics import precision_recall_curve, classification_report, roc_auc_score
import xgboost as xgb

In [ ]:
# --- GCP Configuration ---
PROJECT_ID = "dh-inf-for-ai-509122"
BUCKET_NAME = "milestone_one_data"
LOCATION = "EU"

# --- Authentication options ---
# Option A: Google Colab / Vertex AI:
from google.colab import auth
auth.authenticate_user()

print(f"Target project: {PROJECT_ID}")
print(f"Target bucket: gs://{BUCKET_NAME}")

In [ ]:
# Google Cloud Storage Bucket Setup
def create_gcs_bucket(bucket_name, project_id, location="EU"):
    """Create a GCS bucket if it does not already exist."""
    client = storage.Client(project=project_id)

    if client.lookup_bucket(bucket_name):
        print(f"Bucket 'gs://{bucket_name}' already exists.")
        return

    bucket = client.bucket(bucket_name)
    bucket.iam_configuration.uniform_bucket_level_access_enabled = True
    bucket.versioning_enabled = True
    bucket.lifecycle_rules.add_delete_rule(
        age=7,                       # Number of days
        is_live=False                # False means target only archived historical versions
    )
    bucket = client.create_bucket(bucket, location=location)
    print(f"Successfully created bucket: gs://{bucket.name}")


create_gcs_bucket(BUCKET_NAME, PROJECT_ID, LOCATION)

In [ ]:
import os
from datetime import date
import kagglehub
from google.colab import userdata
userdata.get('kaggle')

# Download the dataset folder
dataset_folder = kagglehub.dataset_download("nudratabbas/healthcare-fraud-detection-dataset")
print("Dataset folder path:", dataset_folder)

# List the files to find the exact CSV name
files = os.listdir(dataset_folder)
print("Files in folder:", files)

# Get the list of all files in the folder
files_list = os.listdir(dataset_folder)

# Loop through and print each file name
for file in files_list:
    print(file)

# Construct the full local path to your CSV file
# (Assuming the file inside is named 'healthcare-fraud-detection-dataset.csv' or similar)
csv_filename = files[0]  # This automatically grabs the first file in the folder
DATASET_LOCAL_RAW_PATH = os.path.join(dataset_folder, csv_filename)
print("Full local path to file:", DATASET_LOCAL_RAW_PATH)


# 5. Define your GCS upload function
def upload_to_gcs(bucket_name, source_file_name, destination_blob_name):
    """Upload a local file to a GCS object path."""
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(source_file_name)
    print(f"Uploaded {source_file_name} -> gs://{bucket_name}/{destination_blob_name}")

# 6. Execute the upload
# Replace 'YOUR_BUCKET_NAME_HERE' with your actual bucket variable or string
RETRIEVAL_DATE = date.today().isoformat()
upload_to_gcs(BUCKET_NAME, DATASET_LOCAL_RAW_PATH, f"{RETRIEVAL_DATE}/raw/healthcare_fraud.csv")

In [ ]:
df_health_fraud = pd.read_csv(DATASET_LOCAL_RAW_PATH)

print("Dataset shape:", df_health_fraud.shape)
display(df_health_fraud.head())

In [ ]:
df_health_fraud.dtypes

In [ ]:
df_health_fraud.shape

In [ ]:
df_health_fraud.describe()

In [ ]:
pd.isna(df_health_fraud)

In [ ]:
df_health_fraud.mean(numeric_only=True)

In [ ]:
# Display class distribution
class_distribution = df_health_fraud['Is_Fraud'].value_counts(normalize=True) * 100
print("Class Distribution (%):")
print(class_distribution)

In [ ]:
# Statistics for transaction amounts
transaction_amount_stats = df_health_fraud['Approved_Amount'].describe()
print("\nStatistics for Transaction Amounts:")
print(transaction_amount_stats)

In [ ]:
# Check and handle missing values
print("Missing values before handling:")
print(df_health_fraud.isnull().sum())

In [ ]:
df_health_fraud.isnull().sum().values

In [ ]:
df_health_fraud.duplicated().sum()

In [ ]:
#data skew
skew =df_health_fraud.select_dtypes([np.number]).skew().sort_values(ascending =False )
skew_df= pd.DataFrame({'skew':skew})
skew_df.head(10)

In [ ]:
#variance
var= df_healthcare.select_dtypes([np.number]).var().sort_values(ascending =True )
var_df= pd.DataFrame({'var':var})
var_df.head(10)

In [ ]:
#Visual Exploratory
sns.pairplot(df_health_fraud.iloc[0:10000], hue= 'Is_Fraud')

In [ ]:
plot = klib.corr_plot(df_healthcare,  figsize=(12,10))

In [ ]:
klib.corr_plot(df_healthcare, split='pos') # displaying only positive correlations, other settings include threshold, cmap...
#klib.corr_plot(df_healthcare, split='neg') # displaying only negative correlations

In [ ]:
#Zooming on the correlation between numerical variables and target.
df_healthcare.select_dtypes([np.number]).corr()['Is_Fraud'][:-1].plot.barh(figsize=(8,6),alpha=.6,color='darkblue')

In [ ]:
# Data cleaning. Value unknown is least destructive option as it does not artificially inflate other categories when the category is not know. It is usuful in cases where this might actually indicate a fraud pattern
# Instead of dropping, fill missing text columns with 'Unknown'
df_health_fraud['Insurance_Type'] = df_health_fraud['Insurance_Type'].fillna('Unknown')
df_health_fraud['Provider_Specialty'] = df_health_fraud['Provider_Specialty'].fillna('Unknown')

# Verify the changes
print(df_health_fraud['Insurance_Type'].value_counts())
print(df_health_fraud['Provider_Specialty'].value_counts())

In [ ]:
# Analysis suggests values are MCAR. It is also common in cases where the patient has not had prior visits for this field to be blank instead of '0' prior visits being recorded. Seems an okay option to change to 0 for these.
# Fill missing numerical visits with 0
df_health_fraud['Prior_Visits_12m'] = df_health_fraud['Prior_Visits_12m'].fillna(0)

# Verify the changes
print(df_health_fraud['Prior_Visits_12m'].value_counts())

In [ ]:
# Check and handle missing values
print("Missing values before handling:")
print(df_health_fraud.isnull().sum())

In [ ]:
# Transform raw Claim_Submission_Date into safe, leak-free operational features
submission_datetime = pd.to_datetime(df_health_fraud['Claim_Submission_Date'])
df_health_fraud['Submission_Month'] = submission_datetime.dt.month
df_health_fraud['Is_Submission_Weekend'] = submission_datetime.dt.weekday.isin([5, 6]).astype(int)

# =====================================================================
# 2. STRICT PARTITIONING: Train (80%) / Dev (10%) / Test (10%)
# =====================================================================
train_df, temp_df = train_test_split(
    df_health_fraud,
    test_size=0.20,
    random_state=42,
    stratify=df_health_fraud["Is_Fraud"],
)

dev_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["Is_Fraud"],
)

# Tag partitions for cloud tracking audit trails
train_df['split'] = 'train'
dev_df['split'] = 'dev'
test_df['split'] = 'test'

# =====================================================================
# 3. BEHAVIORAL PROFILE FEATURE ENGINEERING (Strictly on Train Folds)
# =====================================================================
print("Engineering safe provider historical velocity profiles...")

# Calculate lag metrics using ONLY the training rows to prevent lookahead leakage
provider_velocity_stats = train_df.groupby('Provider_ID').agg(
    Hist_Pct_Fast_Claims=('Days_Between_Service_and_Claim', lambda x: (x < 5).mean()),
    Hist_Mean_Lag=('Days_Between_Service_and_Claim', 'mean'),
    Hist_Lag_Std=('Days_Between_Service_and_Claim', 'std')
).reset_index()

# Fix variance for providers appearing exactly once in training history
provider_velocity_stats['Hist_Lag_Std'] = provider_velocity_stats['Hist_Lag_Std'].fillna(0)

# Establish global fallbacks for new providers unseen during the training window
global_pct_fast = train_df['Days_Between_Service_and_Claim'].apply(lambda x: x < 5).mean()
global_mean_lag = train_df['Days_Between_Service_and_Claim'].mean()
global_std_lag = train_df['Days_Between_Service_and_Claim'].std()

def apply_historical_velocity(target_df, stats_df):
    output_df = target_df.merge(stats_df, on='Provider_ID', how='left')
    output_df['Hist_Pct_Fast_Claims'] = output_df['Hist_Pct_Fast_Claims'].fillna(global_pct_fast)
    output_df['Hist_Mean_Lag'] = output_df['Hist_Mean_Lag'].fillna(global_mean_lag)
    output_df['Hist_Lag_Std'] = output_df['Hist_Lag_Std'].fillna(global_std_lag)
    return output_df

# Apply lookups cleanly across tracking data splits
train_df = apply_historical_velocity(train_df, provider_velocity_stats)
dev_df = apply_historical_velocity(dev_df, provider_velocity_stats)
test_df = apply_historical_velocity(test_df, provider_velocity_stats)

# Re-combine dataframes with all engineered features for permanent storage
PROCESSING_DATE = date.today().isoformat()
df_with_splits = pd.concat([train_df, dev_df, test_df])
LOCAL_SPLITS_PATH = "healthcare_fraud_splits.csv"
df_with_splits.to_csv(LOCAL_SPLITS_PATH, index=False)
upload_to_gcs(BUCKET_NAME, LOCAL_SPLITS_PATH, f"{PROCESSING_DATE}/splits/healthcare_fraud_splits.csv")

# --- SAVE AS PARQUET AND UPLOAD TO GCS ---
LOCAL_PARQUET_PATH = "healthcare_fraud_features.parquet"

# Save locally to parquet format (requires 'pyarrow' or 'fastparquet' installed)
df_with_splits.to_parquet(LOCAL_PARQUET_PATH, index=False)

# Upload the optimized feature file to a features directory in your GCS bucket
upload_to_gcs(BUCKET_NAME, LOCAL_PARQUET_PATH, f"{PROCESSING_DATE}/features/healthcare_fraud_features.parquet")